# Attention and the transformer

The RNN compresses an entire sequence into a single hidden vector before making predictions. For long sequences that bottleneck loses early context. Attention removes it: every output position can directly query every input position, and the model learns which positions are relevant. This notebook builds scaled dot-product attention, multi-head attention, positional encoding, and a transformer encoder block from scratch in PyTorch.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"device: {device}")

## 1. Intuition

An RNN reads left to right, updating a hidden state h_t at each step. By the end of a 500-token sentence, h_T must carry everything relevant from position 1. That is a lot to ask of a fixed-size vector.

Attention sidesteps this by letting each output position issue a **query** against all input positions at once. Each input position has a **key** (what it offers) and a **value** (what it contributes). The output is a weighted sum of values, where the weights come from how well the query matches each key. Nothing is compressed; every position stays accessible.

## 2. Scaled dot-product attention

Given matrices Q (queries), K (keys), V (values) all of dimension `(seq_len, d_k)`:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

The dot product QK^T measures similarity. Dividing by $\sqrt{d_k}$ keeps the values in a range where the softmax gradient is healthy — without it, large d_k causes the dot products to grow proportionally, pushing softmax into saturation and killing gradients.

In [ ]:
# NumPy walkthrough on a 4-token sequence
rng = np.random.default_rng(SEED)
d_k = 8
seq_len = 4

Q_np = rng.standard_normal((seq_len, d_k))
K_np = rng.standard_normal((seq_len, d_k))
V_np = rng.standard_normal((seq_len, d_k))

scores = Q_np @ K_np.T / math.sqrt(d_k)          # (4, 4)
weights = np.exp(scores) / np.exp(scores).sum(axis=-1, keepdims=True)  # softmax
output_np = weights @ V_np                        # (4, d_k)

print("attention weights (rows sum to 1):")
print(np.round(weights, 3))
print("\noutput shape:", output_np.shape)

In [ ]:
def scaled_dot_product_attention(
    q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, mask: torch.Tensor | None = None
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (output, attention_weights). Inputs: (..., seq, d_k)."""
    d_k = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)  # (..., seq, seq)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    return torch.matmul(weights, v), weights


# quick check
q = torch.randn(1, seq_len, d_k)
k = torch.randn(1, seq_len, d_k)
v = torch.randn(1, seq_len, d_k)
out, attn = scaled_dot_product_attention(q, k, v)
print("output shape:", out.shape)    # (1, 4, 8)
print("weights sum:", attn.sum(-1))  # all 1.0

## 3. Multi-head attention

Instead of one set of Q, K, V projections, run `h` sets in parallel. Each **head** projects the input into a lower-dimensional subspace (d_k = d_model // h), computes attention independently, then concatenates the results and projects back to d_model. Different heads learn to attend to different relationships — one might track subject-verb agreement, another coreference.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int) -> None:
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)

    def forward(
        self, x: torch.Tensor, mask: torch.Tensor | None = None
    ) -> tuple[torch.Tensor, torch.Tensor]:
        batch, seq, _ = x.shape
        # project and split into heads
        def split_heads(t: torch.Tensor) -> torch.Tensor:
            return t.view(batch, seq, self.n_heads, self.d_k).transpose(1, 2)  # (B, h, S, d_k)

        q, k, v = split_heads(self.w_q(x)), split_heads(self.w_k(x)), split_heads(self.w_v(x))
        attn_out, weights = scaled_dot_product_attention(q, k, v, mask)  # (B, h, S, d_k)
        # concatenate heads
        concat = attn_out.transpose(1, 2).contiguous().view(batch, seq, -1)  # (B, S, d_model)
        return self.w_o(concat), weights


d_model, n_heads = 32, 4
mha = MultiHeadAttention(d_model, n_heads)
x_test = torch.randn(2, 10, d_model)  # batch=2, seq=10
out_mha, w_mha = mha(x_test)
print("MHA output shape:", out_mha.shape)   # (2, 10, 32)
print("attention weights shape:", w_mha.shape)  # (2, 4, 10, 10)

## 4. Positional encoding

Attention is order-invariant: shuffle the tokens and the dot products are the same. To give the model a sense of position, add a fixed signal to each token embedding before feeding it in. The original transformer uses sinusoids:

$$PE_{(pos,\,2i)} = \sin\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right), \quad PE_{(pos,\,2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

Even dimensions get a sine, odd dimensions a cosine. The wavelengths range from $2\pi$ (dimension 0) to $20000\pi$ (last dimension), so each position has a unique fingerprint across the full depth of the model.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512) -> None:
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1)]


pos_enc = PositionalEncoding(d_model, max_len=50)
pe_matrix = pos_enc.pe.squeeze(0).detach().numpy()  # (50, 32)

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(pe_matrix.T, aspect="auto", cmap="RdBu", origin="lower")
ax.set_xlabel("position")
ax.set_ylabel("dimension")
ax.set_title("Sinusoidal positional encoding (d_model=32, 50 positions)")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

Low-numbered dimensions oscillate quickly (short wavelength); high-numbered dimensions change slowly. The model can read off both fine-grained and coarse position information from the same vector.

## 5. Transformer encoder block

One encoder block combines multi-head self-attention with a position-wise feed-forward network. Both sub-layers use a residual connection (Add) followed by layer normalisation. The residual ensures gradients flow directly to earlier layers without passing through the attention computation.

Block: `x → MHA(x) + x → LayerNorm → FFN(x) + x → LayerNorm`

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class EncoderBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1) -> None:
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ff = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(
        self, x: torch.Tensor, mask: torch.Tensor | None = None
    ) -> tuple[torch.Tensor, torch.Tensor]:
        attn_out, weights = self.attn(x, mask)
        x = self.norm1(x + self.drop(attn_out))
        x = self.norm2(x + self.drop(self.ff(x)))
        return x, weights


class TransformerEncoder(nn.Module):
    def __init__(
        self, vocab_size: int, d_model: int, n_heads: int, d_ff: int, n_layers: int
    ) -> None:
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model)
        self.blocks = nn.ModuleList(
            [EncoderBlock(d_model, n_heads, d_ff) for _ in range(n_layers)]
        )

    def forward(
        self, x: torch.Tensor, mask: torch.Tensor | None = None
    ) -> tuple[torch.Tensor, list[torch.Tensor]]:
        x = self.pos_enc(self.embed(x))
        all_weights = []
        for block in self.blocks:
            x, w = block(x, mask)
            all_weights.append(w)
        return x, all_weights


encoder = TransformerEncoder(vocab_size=50, d_model=d_model, n_heads=n_heads, d_ff=64, n_layers=2)
print(encoder)
n_params = sum(p.numel() for p in encoder.parameters())
print(f"\nparameters: {n_params:,}")

## 6. Demonstration

Pass a short toy sentence through the encoder. The output has the same shape as the input — the encoder transforms token representations, it does not change the sequence length. Then visualise the attention weights from the first head of the first layer.

In [ ]:
torch.manual_seed(SEED)

# toy sentence: "the cat sat on the mat near the door"
tokens = ["the", "cat", "sat", "on", "the", "mat", "near", "the", "door"]
vocab = {w: i for i, w in enumerate(sorted(set(tokens)))}
token_ids = torch.tensor([[vocab[t] for t in tokens]])  # (1, 9)

encoder.eval()
with torch.no_grad():
    output, all_weights = encoder(token_ids)

print("input shape: ", token_ids.shape)   # (1, 9)
print("output shape:", output.shape)      # (1, 9, 32) — same seq len, richer representations

In [ ]:
# attention weights from layer 0, head 0
w_layer0 = all_weights[0]              # (1, n_heads, seq, seq)
w_head0 = w_layer0[0, 0].numpy()      # (9, 9)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(w_head0, cmap="Blues", vmin=0, vmax=w_head0.max())
ax.set_xticks(range(len(tokens)))
ax.set_yticks(range(len(tokens)))
ax.set_xticklabels(tokens, rotation=45, ha="right")
ax.set_yticklabels(tokens)
ax.set_title("Attention weights — layer 0, head 0")
ax.set_xlabel("attended to (key)")
ax.set_ylabel("query")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

Each row is a query token; each column is the key it attends to. A bright cell at (row i, col j) means token i draws heavily from token j when building its representation. Without training the weights are roughly uniform — trained weights develop structure that reflects syntax and semantics.

## 7. RNN vs transformer

| Property | RNN / LSTM | Transformer encoder |
|---|---|---|
| Parallelism | Sequential — position t depends on t-1 | Fully parallel across positions |
| Context window | Unlimited in theory; fades in practice | Fixed by max_len, accessed directly |
| Memory cost | O(n) | O(n²) in attention weights |
| Positional awareness | Built in by recurrence | Must be added explicitly |
| Parameter scaling | Hidden state grows by 4× for LSTM gates | Stack more layers, widen d_model |

## 8. What breaks

**Quadratic cost.** The attention matrix is `(seq_len, seq_len)`. For a 1,000-token document that is a million values per head per layer. Efficient attention variants (Longformer, FlashAttention) address this but add complexity.

**Fixed positional encoding.** The sinusoidal encoding is deterministic and does not generalise to sequences longer than `max_len`. Relative position encodings (RoPE, ALiBi) used in modern LLMs handle this better.

**No recurrence means no streaming.** The encoder needs the full sequence before it can produce any output. An autoregressive decoder processes left-to-right using a causal mask, but the encoder as built here cannot.

Section 07 picks up here: how pre-trained transformer encoders (BERT) and decoders (GPT) are fine-tuned on downstream tasks.